# Aggregate the Results for a set of Sweeps

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import seaborn as sns

from notebook_utils import parse_tensorboard_logs, aggregate_test_metrics  # ty:ignore[unresolved-import]

SWEEPS_DIR = Path("/project/ultrabench2/20260518-urfm-ultrasam-results/sweeps/")

## Parse Tensorboard Logs

In [ ]:
results = parse_tensorboard_logs(SWEEPS_DIR)
results.head()

In [ ]:
classification_results = results[results["dir_name"].str.contains("classification")]
segmentation_results = results[results["dir_name"].str.contains("segmentation")]

In [ ]:
# Group results by dir_name and keep only the rows with the lowest
# `Epoch/val_loss` value for each group
classification_results = (
    classification_results.sort_values("Epoch/val_loss")
    .groupby("dir_name")
    .first()
    .reset_index()
)

segmentation_results = (
    segmentation_results.sort_values("Epoch/val_loss")
    .groupby("dir_name")
    .first()
    .reset_index()
)

## Calculate Best Validation Statistics Across Runs

In [ ]:
# Group the classification results by `model`, `task`, `shot_pct` and `mode`
# and compute the mean and std. dev. of `Epoch/val_loss` and `Epoch/val_acc`
# for each group
classification_summary = (
    classification_results.groupby(["task", "model", "shot_pct", "mode"])
    .agg(
        {
            "Epoch/val_loss": ["mean", "std"],
            "Epoch/val_acc": ["mean", "std"],
        }
    )
    .reset_index()
)
classification_summary.round(4)

In [ ]:
# Group the segmentation results by `model`, `task`, `shot_pct` and `mode`
# and compute the mean and std. dev. of `Epoch/val_loss` and `Epoch/val_dice`
# for each group
segmentation_summary = (
    segmentation_results.groupby(["task", "model", "shot_pct", "mode"])
    .agg(
        {
            "Epoch/val_loss": ["mean", "std"],
            "Epoch/val_dice": ["mean", "std"],
        }
    )
    .reset_index()
)
segmentation_summary.round(4)

## Calculate Test Metric Statistics

In [ ]:
test_results = aggregate_test_metrics(SWEEPS_DIR)
test_results.head()

In [ ]:
# Group the classification results by `model`, `task`, `shot_pct` and `mode`
# and compute the mean and std. dev. of `Epoch/val_loss` and `Epoch/val_acc`
# for each group
classification_test_results = test_results[
    test_results["task"].str.contains("classification")
]

classification_test_summary = (
    classification_test_results.groupby(["task", "model", "shot_pct", "mode"])
    .agg(
        {
            "accuracy": ["mean", "std"],
            "balanced_accuracy": ["mean", "std"],
            "f1_macro": ["mean", "std"],
            "f1_weighted": ["mean", "std"],
            "mcc": ["mean", "std"],
        }
    )
    .reset_index()
)
classification_test_summary.round(4)

In [ ]:
# Group the segmentation results by `model`, `task`, `shot_pct` and `mode`
# and compute the mean and std. dev. of `Epoch/val_loss` and `Epoch/val_dice`
# for each group
segmentation_test_results = test_results[
    test_results["task"].str.contains("segmentation")
]

segmentation_test_summary = (
    segmentation_test_results.groupby(["task", "model", "shot_pct", "mode"])
    .agg(
        {
            "dice": ["mean", "std"],
            "positive_dice": ["mean", "std"],
            "iou": ["mean", "std"],
        }
    )
    .reset_index()
)
segmentation_test_summary.round(4)

In [ ]:
# Plot the classification test results
x_tick_labels = ["AUL", "Butterfly", "GBCU", "MASLD", "MMOTU"]

g = sns.catplot(
    data=classification_test_results,
    x="task",
    y="balanced_accuracy",
    errorbar="sd",
    hue="model",
    kind="bar",
    order=sorted(classification_test_results["task"].unique()),
)
g.set_axis_labels("Task", "Balanced Accuracy")
g.set_xticklabels(x_tick_labels)

plt.show()

In [ ]:
# Plot the segmentation test results
x_tick_labels = [
    "AUL (Liver)",
    "AUL (Mass)",
    "Open Kidney (Capsule)",
    "Open Kidney (Regions)",
    "PSFHS",
]

g = sns.catplot(
    data=segmentation_test_results,
    x="task",
    y="positive_dice",
    errorbar="sd",
    hue="model",
    kind="bar",
    order=sorted(segmentation_test_results["task"].unique()),
)
g.set_axis_labels("Task", "Positive Dice Score")
g.set_xticklabels(x_tick_labels, rotation=45, ha="right")

plt.show()